# DarkSight on Kaggle — full pipeline (2× T4)

**Notebook settings (right-hand panel):** Accelerator = **GPU T4 x2** · Internet = **On** · Persistence = Variables and Files.

**Data:** attach an ExDark dataset that contains **both images and the bbGt annotations**
(Add Input → search "ExDark"). The next cells find it automatically and tell you if annotations are missing.
If your mirror has no annotations, upload the official 5 MB `ExDark_Annno.zip` as your own dataset and attach that too.

**Where things live:** datasets are built in `/kaggle/temp` (big, wiped at session end), while runs, weights and
results go to `/kaggle/working` (20 GB, downloadable / saved as output).

Expected wall-clock on 2× T4: prepare ~5 min · enhance ~10 min · each detector ~45–75 min · evaluation ~10 min.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!free -g | head -2; df -h /kaggle/working /kaggle/temp | tail -2

In [ ]:
import os
os.makedirs('/kaggle/temp', exist_ok=True)
%cd /kaggle/working
!git clone https://github.com/<your-team>/darksight.git darksight 2>/dev/null || (cd darksight && git pull)
%cd /kaggle/working/darksight
!pip -q install "ultralytics>=8.3,<9" scikit-image
# datasets live on the big scratch disk; results stay in /kaggle/working
os.environ['DARKSIGHT_DATA'] = '/kaggle/temp/exdark_yolo'
import torch, ultralytics; print(torch.__version__, ultralytics.__version__, torch.cuda.device_count(), 'GPUs')

## 1 · Find and check the attached ExDark data

In [ ]:
!python scripts/find_exdark.py /kaggle/input

In [ ]:
# paths chosen above (edit manually if you attached several mirrors)
IMAGES = !python scripts/find_exdark.py /kaggle/input --print images
ANNOTS = !python scripts/find_exdark.py /kaggle/input --print annotations
IMAGES, ANNOTS = IMAGES[-1].strip(), ANNOTS[-1].strip()
print(IMAGES, '|', ANNOTS)

## 2 · Convert to YOLO format (official 3000/1800/2563 split)

In [ ]:
!python scripts/prepare_exdark.py --images "{IMAGES}" --annotations "{ANNOTS}" \
    --out /kaggle/temp/exdark_yolo/raw --meta /kaggle/temp/exdark_meta.csv \
    --report /kaggle/working/darksight/reports/data_stats.md --workers 4
!cp /kaggle/temp/exdark_meta.csv data/exdark_meta.csv
!python scripts/visualize_labels.py --root /kaggle/temp/exdark_yolo/raw --split train --n 12 --out /kaggle/working/label_check

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/label_check/train/*.png'))[:4]:
    display(Image(p, width=420))   # boxes must sit on the objects before training anything

## 3 · Zero-DCE enhanced copy of the whole dataset

In [ ]:
!python scripts/enhance_dataset.py --src /kaggle/temp/exdark_yolo/raw --dst /kaggle/temp/exdark_yolo/enhanced --half
!python scripts/verify_dataset.py --a /kaggle/temp/exdark_yolo/raw --b /kaggle/temp/exdark_yolo/enhanced

## 4 · Train the two detectors

`configs/train.yaml` is the single source of truth; the only overrides are the 2-GPU ones
(`--device 0,1 --batch 32` keeps 16 images per GPU, exactly like the single-GPU config).
**Both detectors must get identical flags** — that is what makes E2 vs E4 a fair comparison.

In [ ]:
# smoke test first (~3 min): proves data + DDP work before spending GPU hours
!python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/raw --name smoke \
    --model yolov8n.pt --epochs 1 --fraction 0.1 --device 0,1 --batch 32 --project /kaggle/temp/runs_smoke

In [ ]:
!python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/raw --name det_raw \
    --device 0,1 --batch 32 --project /kaggle/working/runs

In [ ]:
!python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/enhanced --name det_enh \
    --device 0,1 --batch 32 --project /kaggle/working/runs
!cp weights/det_raw.pt weights/det_enh.pt /kaggle/working/

**If the session dies mid-training:** re-run cells 1–3 (fast), then
`!python scripts/train_detector.py --resume /kaggle/working/runs/det_raw/weights/last.pt`.
Checkpoints are written every 5 epochs to `/kaggle/working`, which survives a session restart when
"Persistence: Files" is on.

## 5 · Experiments E0–E5

In [ ]:
# gated variant for E5 — tau is chosen on val, then applied to test
!python scripts/enhance_dataset.py --src /kaggle/temp/exdark_yolo/raw --dst /kaggle/temp/exdark_yolo/enhanced_gated \
    --gate-tau 0.35 --splits val test --half
!python scripts/run_experiment.py --exp E0 E1 E2 E3 E4 E5 --device 0

In [ ]:
!python scripts/compare_experiments.py --a E2 --b E4 --n 1000
!python scripts/compare_experiments.py --a E2 --b E3 --n 1000
!python scripts/eval_by_condition.py --exp E0 E1 E2 E3 E4 E5
!python scripts/make_results_table.py
!python scripts/make_qualitative.py --raw-exp E2 --enh-exp E4 --pick gain --n 6
!python scripts/make_qualitative.py --raw-exp E2 --enh-exp E4 --pick loss --n 3
!python scripts/benchmark_latency.py --detector weights/det_enh.pt --device 0

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary_test.md').read()))
for p in ['results/figures/overall_test.png', 'results/figures/per_class_test.png', 'results/figures/by_light_test.png']:
    display(Image(p, width=760))

## 6 · Save everything you need off Kaggle

In [ ]:
# rendered detections for the report/demo (test split, both pipelines)
!python scripts/render_predictions.py --exp E2 E4 --side-by-side E2 E4 --conf 0.25
!cd /kaggle/working/darksight && zip -q -r /kaggle/working/darksight_results.zip results reports weights/det_*.pt -x "results/E*/predictions_*.jsonl"
!ls -lh /kaggle/working/darksight_results.zip /kaggle/working/det_*.pt

Download `darksight_results.zip` and `det_raw.pt` / `det_enh.pt` from the notebook's **Output** panel,
then commit the metrics JSON/CSV to the repo (weights stay out of git) and run the demo locally:
`python app/app.py`.